In [1]:
%cd /home/smalani/PartialObservations

from main import utils
# import train_model_filename

/home/smalani/PartialObservations


In [2]:
from main import preprocess
from main.utils import Network, MLP, Model_Train, progress
# import preprocess
# from BandFModel import datagen#, make_plots, helper
# from config import config

In [3]:
mlp1 = utils.MLP(1,[2],1)
mlp2 = utils.MLP(1,[2],1)

for param in mlp1.parameters():
    print(param)
print('=============')
for param in mlp2.parameters():
    print(param)

Parameter containing:
tensor([[0.3316],
        [0.3257]], requires_grad=True)
Parameter containing:
tensor([0., 0.], requires_grad=True)
Parameter containing:
tensor([[-0.0087, -1.0200]], requires_grad=True)
Parameter containing:
tensor([0.], requires_grad=True)
Parameter containing:
tensor([[1.7269],
        [0.2137]], requires_grad=True)
Parameter containing:
tensor([0., 0.], requires_grad=True)
Parameter containing:
tensor([[ 0.5461, -0.5778]], requires_grad=True)
Parameter containing:
tensor([0.], requires_grad=True)


In [4]:
from config import config
import numpy as np
from main.utils import Network, MLP
import torch

f = np.load("/home/smalani/PartialObservations/minmax/minmax.npz")

xmax = f['arr_0']
xmin = f['arr_1']

print('maxmins')
print(xmax)
print(xmin)


norm_func = lambda input, device: (input - torch.tensor(xmin).float().to(device)) / \
                        ((torch.tensor((xmax - xmin)).float().to(device)) + 1e-10)
inv_norm_func = lambda input, device: input * ((torch.tensor((xmax - xmin)).float().to(device)) + 1e-10) \
                                + torch.tensor(xmin).float().to(device)


if config["MODEL"]["BOX"] == 'Black':
    # Create the network architecture
    mlp = MLP(6, config["MODEL"]["NUM_HIDDEN"], 6)
    
    class my_Network(Network):
        def __init__(self, network, train_size, xdim, norm_func=lambda input, device: input,
                        inv_norm_func=lambda input, device: input, init_available=True, device=None, 
                        tf_prop=1., integrator='RK4', add_par_num=0):
            super(my_Network, self).__init__(network, train_size, xdim, norm_func,
                        inv_norm_func, init_available, device, 
                        tf_prop, integrator, add_par_num)

            self.additional_pars = torch.nn.Parameter((torch.zeros(6)-1).to(self.device), requires_grad = True) 

        def output(self, x, par):

            # ANN_input = torch.cat((self.norm_func(x), par/20), dim=-1)
            ANN_input = self.norm_func(x)
            out = self.net(ANN_input)
            out = self.inv_norm_func(out) * (100 ** (self.additional_pars))

            return out

elif config["MODEL"]["BOX"] == 'Grey' or config["MODEL"]["BOX"] == 'Gray':
    
    # Create the network architecture
    mlp = MLP(6, config["MODEL"]["NUM_HIDDEN"], 3)
    
    if config["MODEL"]["Parameters"] == 'Trainable':
        class my_Network(Network):
            def __init__(self, network, train_size, xdim, norm_func=lambda input, device: input,
                            inv_norm_func=lambda input, device: input, init_available=True, device=None, 
                            tf_prop=1., integrator='RK4', add_par_num=2):
                super(my_Network, self).__init__(network, train_size, xdim, norm_func,
                            inv_norm_func, init_available, device, 
                            tf_prop, integrator, add_par_num)

                self.additional_pars = torch.nn.Parameter((torch.cat(((torch.tensor([0.4476, -0.4859, -0.6419])), 
                                                                        (torch.zeros(4) + 1)))).to(self.device), 
                                                            requires_grad = True)

            # def output(self, x_input, par):

            #     ANN_input = self.norm_func(x_input)[...,[0,2,3,4,5]]
            #     ANN_output = self.net(ANN_input) * (2 ** (7 * self.additional_pars[...,:3]))

            #     x, y, z, u, v, g = torch.unbind(x_input, dim=-1)
            #     u1_prime, u2_prime, u3_prime = torch.unbind(ANN_output, dim=-1)
                
            #     omega = self.additional_pars[-4] * 10
            #     sigma = self.additional_pars[-3] * 10
            #     rho = self.additional_pars[-2]
            #     eta = self.additional_pars[-1] * 10

            #     alpha, uf, _, _, _, _, _, _, uc1_prime, uc2_prime, uc3_prime = datagen.par_fun()

            #     output = []

            #     output.append(-alpha * x + u1_prime * x - uc1_prime * x)
            #     output.append(-alpha * y + u2_prime * y - uc2_prime * y)
            #     output.append(-alpha * z + u3_prime * z - uc3_prime * z)
            #     output.append(alpha * (uf - u) - u1_prime * x)
            #     output.append(-alpha * v + omega * u1_prime * x - u2_prime * y - sigma * u3_prime * z)
            #     output.append(-alpha * g + rho * u2_prime * y + eta * u3_prime * z)


            #     out = torch.stack((output), dim=-1)
            #     return out

            def output(self, x_input, par):
                ANN_input_out = self.norm_func(x_input)
                ANN_input = torch.clip(ANN_input_out, min=-1., max=2.)
                ANN_output = self.net(ANN_input)
                ANN_output = ANN_output * (100 ** (self.additional_pars[:3]))

                x, y, z, u, v, g = torch.unbind(x_input, dim=-1)

                u1_prime_x, u2_prime_y, u3_prime_z = torch.unbind(ANN_output, dim=-1)                

                omega = self.additional_pars[-4] * 10
                sigma = self.additional_pars[-3] * 10
                rho = self.additional_pars[-2] / 10
                eta = self.additional_pars[-1]

                alpha, uf, _, _, _, _, _, _, uc1_prime, uc2_prime, uc3_prime = datagen.par_fun()

                output = []

                output.append(-alpha * x + u1_prime_x - uc1_prime * x)
                output.append(-alpha * y + u2_prime_y - uc2_prime * y)
                output.append(-alpha * z + u3_prime_z - uc3_prime * z)
                output.append(alpha * (uf - u) - u1_prime_x)
                output.append(-alpha * v + omega * u1_prime_x - u2_prime_y - sigma * u3_prime_z)
                output.append(-alpha * g + rho * u2_prime_y + eta * u3_prime_z)

                out = torch.stack((output), dim=-1)
                return out
            def raw_output(self, x_input, par):

                ANN_input = self.norm_func(x_input)
                ANN_output = self.net(ANN_input) * (2 ** (7 * self.additional_pars))

                return ANN_output
    elif config["MODEL"]["Parameters"] == 'Fixed':
        class my_Network(Network):
            def __init__(self, network, train_size, xdim, norm_func=lambda input, device: input,
                            inv_norm_func=lambda input, device: input, init_available=True, device=None, 
                            tf_prop=1., integrator='RK4', add_par_num=2):
                super(my_Network, self).__init__(network, train_size, xdim, norm_func,
                            inv_norm_func, init_available, device, 
                            tf_prop, integrator, add_par_num)

                # self.additional_pars = torch.nn.Parameter((torch.zeros(3)).to(self.device), 
                #                             requires_grad = True)
                self.additional_pars = torch.nn.Parameter((torch.tensor([0.4877, -0.7982, -0.6599])).to(self.device), 
                                                                requires_grad = True)
                # self.additional_pars = torch.nn.Parameter((torch.tensor([0.3789, -0.9911, -0.7526])).to(self.device), 
                #                                                 requires_grad = True)

                

            def output(self, x_input, par):
                
                # print('outputs')
                ANN_input_out = self.norm_func(x_input)
                ANN_input = torch.clip(ANN_input_out, min=-1., max=2.)
                ANN_output_out = self.net(ANN_input)
                # print(ANN_output)
                ANN_output = ANN_output_out * (100 ** (self.additional_pars))
                # print(ANN_output)
                # assert False

                x, y, z, u, v, g = torch.unbind(x_input, dim=-1)

                # u1_prime, u2_prime, u3_prime = torch.unbind(ANN_output, dim=-1)
                u1_prime_x, u2_prime_y, u3_prime_z = torch.unbind(ANN_output, dim=-1)

                # print('The us')
                # print(u1_prime_x)
                # print(u3_prime_z)

                # u2_prime = torch.zeros(u1_prime.shape).to(self.device)
                

                alpha, uf, omega, sigma, rho, eta, phi1, phi2, uc1_prime, uc2_prime, uc3_prime = datagen.par_fun(D=1/7.3, sf=2.5)

                # u2_prime_y = y * phi1 * v / (1+v)

                output = []

                # output.append(-alpha * x + u1_prime * x - uc1_prime * x)
                # output.append(torch.zeros(output[-1].shape).to(self.device))
                # output.append(-alpha * z + u3_prime * z - uc3_prime * z)
                # output.append(alpha * (uf - u) - u1_prime * x)
                # output.append(-alpha * v + omega * u1_prime * x - u2_prime * y - sigma * u3_prime * z)
                # output.append(-alpha * g + rho * u2_prime * y + eta * u3_prime * z)

                output.append(-alpha * x + u1_prime_x - uc1_prime * x)

                output.append(-alpha * y + u2_prime_y - uc2_prime * y)
                # output.append(torch.zeros(output[-1].shape).to(self.device))

                output.append(-alpha * z + u3_prime_z - uc3_prime * z)
                output.append(alpha * (uf - u) - u1_prime_x)
                output.append(-alpha * v + omega * u1_prime_x - u2_prime_y - sigma * u3_prime_z)
                output.append(-alpha * g + rho * u2_prime_y + eta * u3_prime_z)

                out = torch.stack((output), dim=-1)

                # print('grey box fixed output fun')
                # print(ANN_input)
                # print(ANN_output_out)
                # print(ANN_output)
                # print(out)
                # assert False

                return out
            def raw_output(self, x_input, par):

                ANN_input = self.norm_func(x_input)
                ANN_output = self.net(ANN_input) * (2 ** (7 * self.additional_pars))

                return ANN_output

    else:
        raise ValueError("Tell me whether to train the parameters!")
else:
    raise ValueError("Tell me what box to use!")
        

if config["DATA"]["DUP_REVERSE"]:
    network = my_Network(mlp, config["DATA"]["N_TRAIN"]*2, 6, norm_func=norm_func, inv_norm_func=inv_norm_func, 
                    init_available=config["DATA"]["INIT_AVAILABLE"], integrator='RK4')
else:
    network = my_Network(mlp, config["DATA"]["N_TRAIN"], 6, norm_func=norm_func, inv_norm_func=inv_norm_func, 
                    init_available=config["DATA"]["INIT_AVAILABLE"], integrator='RK4')

maxmins
[7.76480999e+01 6.13450077e-02 3.52889567e-01 2.08507400e+02
 2.11354658e+03 8.21362308e-01]
[1.45181035e+01 1.12344629e-04 2.81247634e-01 3.24970859e+01
 3.97881635e+02 6.58996734e-01]


In [5]:
for p in network.parameters():
    print(p)

Parameter containing:
tensor([[0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        ...,
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000]], device='cuda:0',
       requires_grad=True)
Parameter containing:
tensor([ 0.4476, -0.4859, -0.6419,  1.0000,  1.0000,  1.0000,  1.0000],
       device='cuda:0', requires_grad=True)
Parameter containing:
tensor([[ 3.3875e-01, -1.0881e-02, -3.9576e-01,  3.2537e-01, -8.7017e-01,
         -9.2992e-01],
        [-8.5393e-01,  2.4957e-01, -4.1158e-01,  1.2680e-01, -9.2305e-01,
         -6.3006e-02],
        [ 4.1292e-01,  2.2597e-02,  7.5394e-01,  1.4237e-01,  1.5537e-01,
          4.9308e-02],
        [-7.9634e-01,  3.6131e-01, -1.4924e+00, -1.3856e-02, -7.0548e-02,
         -4.3128e-01],
        [-9.3476e-01,  7